# **1.  Introduction:**

Densely populated regions and craft village areas have experienced the concentration of CO2, badly affecting residents' health and lifestyle. Hanoi, the capital of Vietnam, is particularly vulnerable to air pollution due to its high population density, economic growth, and reliance on private motor vehicles (Vanderbloemen, 2025). Many people have been dealt with health concerns, specifically air pollution has been nominated as the biggest cause. PM2.5 can penetrate deep into the lungs and enter the bloodstream, causing respiratory and cardiovascular issues (Chen, 2020). 

However, challenges such as limited sensor coverage and the costs of real-time monitoring create barriers to accurate assessment and public awareness.

This project addresses these issues by developing a machine-learning–based air-quality prediction system combined with an interactive and user-friendly dashboard. The objective is to provide residents with reliable insights into pollution levels, and explore the potential for integration into broader smart-city and environmental-management.

Regarding the lack of public air-quality datasets for Hanoi, our team selected a model dataset from Kaggle — the IAQ Baquba Hospital dataset – University of Diyala | Kaggle. This serves as our initial testing dataset, allowing us to develop and validate our model before applying it to real-world datasets from megacities and densely populated urban areas, in collaboration with environmental institutions.

# **3. Data Science Questions:**

> ### **Question 1 - Which factors best explain the composite Air Quality index?**

## **1. Introduction** 

In modern Building Management Systems (BMS), the "Air Quality" metric is often a composite score - a single number derived from a weighted aggregation of various pollutants to simplify communication with building occupants. However, understanding which underlying factors drive this score is essential for remediation. If the Air Quality index drops, facility managers need to know whether to increase fresh air intake (to dilute CO<sub>2</sub>) or upgrade filtration (to capture PM2.5).

In this project, our team aims to identify which factors best explain the variance in the Air Quality target variable found in the Baqubah Teaching Hospital dataset. To do this, we employ advanced machine learning techniques, specifically Gradient Boosting Machines (GBM) and Recurrent Neural Networks (RNN). These models have the capacity to map complex, non-linear relationships and, in the case of tree-based models, provide explicit "Feature Importance" rankings.

The dataset features utilized here are the full spectrum of environmental readings: CO<sub>2</sub>, TVOC, PM10, PM2.5, CO, O<sub>3</sub>, Temp, Hum, and LDR. The target variable is the Air Quality column.

## **2. Approach** 
Coming up with the raw data, we noticed that there are missing values, potential timestamps misalignments. In order to bring this data into more insightful information, we conducted a cleaning process on this raw dataset.

In [ ]:
# Handling missing values
missing_percentage = (df.isna().sum() / len(df)) * 100
cols_to_drop = missing_percentage[missing_percentage > 50].index
df.drop(columns=cols_to_drop, inplace=True)
df.shape


In [ ]:
# Convert timestamp to datetime and reasample to 5 minutes 
df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
df_5min = df.resample('5T').mean()
# Hhanding missing values if possible:
iaqs = ['CO2', 'TVOC', 'PM10', 'PM2.5', 'CO', 'Air Quality', 'LDR', 'O3', 'Temp', 'Hum']  
df_5min[iaqs] = df_5min[iaqs].interpolate(
    method='time',      
    limit=3,
    limit_direction='both'
)
df_5min[iaqs] = df_5min[iaqs].ffill().bfill()


The approach to identifying the determinants of the Air Quality index combines predictive accuracy with interpretability. We utilize results and code structures from the xgboost_new.ipynb , lightgbm_new.ipynb , catboost_new.ipynb , and lstm_new.ipynb  notebooks.

### **2.1. Modelling strategy**
To ensure the robustness of the findings, we employed disparate modeling architectures:

- Gradient Boosting (XGBoost, CatBoost, LightGBM):

    - Justification: These algorithms construct ensembles of decision trees. They are exceptionally good at handling tabular data and providing feature importance scores based on "Gain" (how much a feature contributes to reducing prediction error) or "Split Count" (how often a feature is used to make a decision). CatBoost is particularly valuable for its symmetric tree structure which reduces overfitting, while XGBoost provides a highly optimized implementation of gradient boosting.

- Deep Learning (LSTM, Bi-LSTM):

    - Justification: Indoor air quality is temporal. The quality of air at minute $t$ is highly dependent on the state at $t-1$. Long Short-Term Memory (LSTM) networks and their Bidirectional variants capture these lag dependencies. While they are "black boxes" regarding direct feature importance compared to trees, their predictive performance ($R^2$, RMSE) validates whether the chosen features contain sufficient information to model the system accurately.

### **2.2. Feature Importance and Evaluation Metrics**
The primary method for answering "which factors explain the index" is Permutation Feature Importance and Tree-based Feature Importance.

*Evaluation Metrics:* We evaluate the models using Root Mean Squared Error (RMSE) and the Coefficient of Determination ($R^2$). A high $R^2$ indicates that the selected features explain a vast majority of the variance in the Air Quality index. The results from SPATIAL(Single_Variable).ipynb provide specific benchmarks for these metrics.

## **3. Analysis**
Before applying the dataset to the model, we attempted to split the datasets into: 80% using for training model, 20% for testing purpose.
### **3.1. Gradient Boosting Model (XGBoost/Catboost/LightGBM)**:
Then, we applied gradient boosting on the cleaned datasets, evaluating the performance of each model by using RMSE and $R^2$ 

In [ ]:
# XGBoost
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
# Catboost
model = cb.CatBoostRegressor(iterations=100, learning_rate=0.1, depth=5, loss_function='RMSE', verbose=0)
# LightGBM
model = lgb.LGBMRegressor(objective='regression', n_estimators=100, learning_rate=0.1, max_depth=5)

# Evaluation model:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

| Metric \ Model | XGBoost | CatBoost | LightGBM |
|--------|---------|----------|----------|
| **RMSE** | 13.74403155073127 | 14.25634374264263 | 13.75634126799002 |
| **R²**   | 0.9311271704164953 | 0.9258969717861947 | 0.9310037445177246 |


Next, we derived feature importance figure afterwards:

In [ ]:
# Feature importance
feature_importance = model.get_feature_importance()
sorted_idx = np.argsort(feature_importance)


<div style="display: flex; justify-content: space-between;">

  <figure style="text-align: center;">
    <img src="xgboost_fi.png" width="320">
    <figcaption><b>XGBoost Feature Importance</b></figcaption>
  </figure>

  <figure style="text-align: center;">
    <img src="catboost_fi.png" width="320">
    <figcaption><b>CatBoost Feature Importance</b></figcaption>
  </figure>

  <figure style="text-align: center;">
    <img src="lightgbm_fi.png" width="320">
    <figcaption><b>LightGBM Feature Importance</b></figcaption>
  </figure>

</div>


**Interpret the results**

- PM2.5 / PM10: Consistently the top features by "Gain". The composite Air Quality index is almost certainly defined by a formula where PM is the dominant term (e.g., US EPA AQI standards).

- LDR: The second-tier feature in both XGBoost and LightGBM. In indoor environments such as a hospital, LDR effectively captures time-of-day and occupancy patterns, both of which correlate strongly with fluctuations in airborne particulate levels.

- Temp/Hum: These appear with lower importance scores. While they modulate the gases, they are not the primary constituents of the AQI formula itself.

### **3.2 Deep Learning model (LSTM/Bi-LSTM)**
To validate the temporal predictability, we examine the LSTM implementation from lstm.ipynb, spatial_BiLSTM.ipynb  and the results from SPATIAL(Single_Variable).ipynb.

In [ ]:
# LSTM Model
model = Sequential()
model.add(LSTM(50, activation='relu', input_shape=(n_input, n_features)))
model.add(Dropout(0.2))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

# Bi-LSTM Model
def stacked_LSTM(X, Y):
    if X.shape[0] == 0:
        print("Không có dữ liệu huấn luyện!")
        return None, None
        
    time_step = X.shape[1]
    input_dim = X.shape[2]
    out = Y.shape[2]
    model = Sequential()
    model.add(Masking(mask_value=-1., input_shape=(time_step, input_dim)))
    model.add(Bidirectional(LSTM(32, activation='relu', return_sequences=True)))
    model.add(Dense(out))
    model.compile(loss='mean_absolute_error', optimizer=keras.optimizers.Adam(learning_rate=1e-5))

    # Add early stopping
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    hist = model.fit(X, Y, epochs=5, validation_split=0.2, verbose=1, batch_size=10, callbacks=[early_stopping])
    model.summary()
    return model, hist

**Note**: Update interpretation for results soon.

## **4. Discussion**
### **4.1 Interpretation**
The comprehensive analysis of the Data Science projects leads to a clear conclusion regarding the composite Air Quality index. It is not a mysterious variable but a deterministic function driven primarily by Particulate Matter ($PM_{2.5}$ and $PM_{10}$). The machine learning models, both Tree-based and Deep Learning, converge on this fact.

### **4.2 Limitations and future work**


### **4.3 Conclusion**
In answer this question, we found that Particulate Matter ($PM_{2.5}$) is the dominant explanatory factor for the Air Quality Index, with LDR playing a critical secondary role.  These pollutants are physically coupled to Temperature and Humidity, which act as thermodynamic accelerators for chemical volatility. The successful application of LSTM and Bi-LSTM models 1 with extremely high accuracy ($R^2 > 0.97$) demonstrates that a predictive Early Warning System (EWS) is technically feasible. Such a system could forecast air quality degradation 10 minutes in advance (based on the "Best Shift" lag), allowing automated Building Management Systems to increase ventilation rates preemptively, thereby securing the health and safety of the hospital's occupants.

# **6. Lifecycle Reflection:**

Throughout this project, our team consistently followed the Data Science Life Cycle to ensure that each stage contributed meaningfully to the development of a reliable air-quality prediction system.

In the problem identification and business understanding phase, we first examined the environmental and public-health concerns associated with PM2.5 and CO₂ exposure in densely populated and craft-village regions. From this, our project has a clear vision: developing a predictive tool and dashboard that would help local residents better monitor air-quality conditions and make informed decisions based on predicted pollution levels. This foundation guided all subsequent phases and ensured that the technical work aligned with a socially relevant need.

After considering the dataset, we choose to go with a dataset on Kaggle with various air quality features. This dataset performs outdoor air quality index, very compatible for our project. The timestamp is up-to-date from 2024, ensuring not an out-dated model.

In the data processing phase, we applied cleaning procedures such as handling missing values, removing duplicates, checking the correlation of CO2, dropping if the correlation higher than 0.9, otherwise interpolating. Dropping columns with more than half missing is necessary. We also create new time-series features, especially creating a 5 minute gap between each data.

We visualize univariate and bivariate to learn the pattern between features like PM2.5 and PM10, CO2 and Temp to notify any relationship. Lineplots of CO2 and PM are presented to analyze the hourly average concentration. We also classify particulate matter into hours and days, and other advanced EDA techniques such as coefficient of features with PM2.5, heatmap of concentration by hour and day of week, and multivariate analysis, specifically with three most concerning factors: CO2, PM2.5 and PM10.
